In [ ]:
import os
import time
from pathlib import Path

import anndata as ad
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from spagrn.regulatory_network import InferNetwork, format_time

# Helper timer (optional)
from contextlib import contextmanager
@contextmanager
def timer(msg: str):
    t0 = time.time()
    print(msg)
    yield
    t1 = time.time()
    print(f"{msg} completed in {format_time(t1 - t0)}\n")
from spagrn.regulatory_network import InferNetwork as irn

In [ ]:
# --------- Configure your inputs here ----------
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_subthalamic nucleus.h5ad")
tfs_fn = 'GRN_resource/hs_hgnc_tfs.txt'
database_fn = 'GRN_resource/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
motif_anno_fn = 'GRN_resource/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl'

niche_human = pd.read_csv('GRN_resource/lr_network_human.csv')
niche_mouse = pd.read_csv('GRN_resource/lr_network_mouse.csv')

output_dir_region = "Output/GRN_output/test_large_function_fast"

# Optional: niche relationships (DataFrame) for Step 6.1
# niches = pd.read_csv("/path/to/niches.csv")
niches = None
receptor_key = "to"

# --------- Parameters aligned to your call ----------
layers = "count"
latent_obsm_key = "align_spatial_3d"
cluster_label = "region_h2"
num_workers = 15
methods = ["FDR_I", "FDR_C", "FDR_G"]
operation = "intersection"
mode = "geary"            # one of: 'moran', 'geary', 'zscore'
model = "danb"
n_neighbors = 10
rho_mask_dropouts = False
local = False
combine = False
somde_k = 20
cache = True
save_tmp = True
weighted_graph = False
umi_counts_obs_key = None
normalize = False


In [ ]:
adata = irn.preprocess(adata, min_genes=10, min_cells=int(len(adata.obs_names)*0.01), min_counts=10, max_gene_num=20000)
X = adata.X
if sp.isspmatrix_csc(X):
    adata.layers['count'] = X.copy()
else:
    if sp.issparse(X):
        adata.layers['count'] = X.tocsc().copy()
    else:
        # dense numpy array -> convert to CSC sparse matrix
        adata.layers['count'] = sp.csc_matrix(X)
del X
print(adata)

In [ ]:
# Pipeline init and sanity checks (mirrors the header in infer)
# ----------------------------------------
print('----------------------------------------')
pipeline_start_time = time.time()

project_name = "split_infer_region"
print(f'Project name is {project_name}')

# Set general output directory
if output_dir_region is None:
    output_dir_region = os.path.dirname(os.path.abspath(__file__))
Path(output_dir_region).mkdir(parents=True, exist_ok=True)
print(f'Saving output files into {output_dir_region}')

# Temporary dir
tmp_dir = os.path.join(output_dir_region, 'tmp_files')
if save_tmp:
    Path(tmp_dir).mkdir(parents=True, exist_ok=True)
    print(f'Saving temporary files to {tmp_dir}')
print('----------------------------------------')

# Sub function test

In [ ]:
# Instantiate InferNetwork
grn = InferNetwork(adata=adata, project_name=project_name)
# Wire tmp_dir used by internal methods
grn.tmp_dir = tmp_dir

# Resolve num_workers / noweights defaults
if num_workers is None:
    from multiprocessing import cpu_count
    num_workers = cpu_count()

noweights = grn.params.get("noweights", False)

exp_mat = grn.data.to_df()

In [ ]:
noweights

In [ ]:
# %%
# Step 1: Loading TF list...
with timer("Step 1: Loading TF list"):
    if tfs_fn is None:
        # Follow repo behavior (but beware: 'all' may cause issues in geary/moran; prefer providing a TF list)
        tfs = 'all'
        print("Loaded all TFs (tfs='all')")
    else:
        if not os.path.exists(tfs_fn):
            raise FileNotFoundError(f"TF list file not found: {tfs_fn}")
        tfs = grn.load_tfs(tfs_fn)
        print(f"Loaded {len(tfs)} TFs")

# %%
# Step 2: Loading ranking databases...
with timer("Step 2: Loading ranking databases"):
    # Note: load_database uses glob.glob(database_dir)
    dbs = grn.load_database(database_fn)
    try:
        ndb = len(dbs)
    except Exception:
        ndb = "unknown"
    print(f"Loaded {ndb} ranking database(s) from pattern: {database_fn}")

In [ ]:
# %%
# Step 3: Starting GRN Inference (spatial gene co-expression analysis)...
with timer("Step 3: GRN Inference"):
    adjacencies = grn.spg(
        data=grn.data,
        gene_list=None,
        tf_list=tfs,
        jobs=num_workers,
        layer_key=layers,
        model=model,
        latent_obsm_key=latent_obsm_key,
        umi_counts_obs_key=umi_counts_obs_key,
        n_neighbors=n_neighbors,
        weighted_graph=weighted_graph,
        cache=cache,
        save_tmp=save_tmp,
        fn=os.path.join(tmp_dir, f'{mode}_adj.csv'),
        local=local,
        methods=methods,
        operation=operation,
        combine=combine,
        mode=mode,
        somde_k=somde_k,
        torch_device='cuda:5',
    )
    print(f"Adjacencies/local correlations shape: {adjacencies.shape if hasattr(adjacencies, 'shape') else 'n/a'}")

# Check result 

## global spatial gene

In [ ]:
more_stats = pd.read_csv("Output/GRN_output/test_large_function_fast/tmp_files/more_stats.csv",sep="\t",index_col=[0])
methods = ["FDR_I", "FDR_C", "FDR_G"]
indices_list = [set(more_stats[more_stats[m] < 0.05].index) for m in methods]
global_inter_genes = set.intersection(*indices_list)
print(f'global spatial gene num (intersection): {len(global_inter_genes)}')
indices_list_pre = indices_list
global_inter_genes_pre = global_inter_genes

In [ ]:
for i in range(3):
    print(len(indices_list[i]))

In [ ]:
more_stats = pd.read_csv("Output/GRN_output/test_large_function/tmp_files/more_stats.csv",sep="\t",index_col=[0])
methods = ["FDR_I", "FDR_C", "FDR_G"]
indices_list = [set(more_stats[more_stats[m] < 0.05].index) for m in methods]
global_inter_genes = set.intersection(*indices_list)
print(f'global spatial gene num (intersection): {len(global_inter_genes)}')

In [ ]:
more_stats

In [ ]:
for i in range(3):
    print(len(indices_list[i]))

In [ ]:
len(set(indices_list[0])&set(indices_list_pre[0]))

In [ ]:
len(set(global_inter_genes)&set(global_inter_genes_pre))

In [ ]:
# %%
# Step 4: Computing co-expression modules...
with timer("Step 4: Computing modules"):
    modules = grn.get_modules(
        adjacencies=adjacencies,
        matrix=exp_mat,
        cache=cache,
        save_tmp=save_tmp,
        rho_mask_dropouts=rho_mask_dropouts
    )
    print(f"Modules created: {len(modules)}")


In [ ]:
params = {
            'rank_threshold': 1500,
            'prune_auc_threshold': 0.05,
            'nes_threshold': 3.0,
            'motif_similarity_fdr': 0.05,
            'auc_threshold': 0.05,
            'noweights': False,
        }
print('Step 5: Running regulons prediction (cisTarget)...')
# ctxcore.genesig.Regulon
regulons = grn.prune_modules(modules,
                              dbs,
                              motif_anno_fn,
                              num_workers=num_workers,
                              save_tmp=save_tmp,
                              cache=cache,
                              fn=os.path.join(tmp_dir, 'motifs.csv'),
                              rank_threshold=params["rank_threshold"],
                              auc_threshold=params["prune_auc_threshold"],
                              nes_threshold=params["nes_threshold"],
                              motif_similarity_fdr=params["motif_similarity_fdr"])

In [ ]:
# 6.0. Cellular Enrichment (aka AUCell)
print('Step 6.0: Calculating cellular enrichment (AUCell)...')
grn.cal_auc(exp_mat,
             regulons,
             auc_threshold=params["auc_threshold"],
             num_workers=num_workers,
             save_tmp=save_tmp,
             cache=cache,
             noweights=noweights,
             normalize=normalize,
             fn=os.path.join(tmp_dir, 'auc_mtx.csv'))

# 7. Calculate Regulon Specificity Scores
print('Step 7: Calculating regulon specificity scores...')
grn.cal_regulon_score(cluster_label=cluster_label, save_tmp=save_tmp,
                       fn=f'{tmp_dir}/regulon_specificity_scores.txt')

In [ ]:
# 8. Save results to h5ad file
print('Step 8: Saving results to h5ad file...')
# dtype=object
output_file = os.path.join(output_dir_region, f'{project_name}_spagrn.h5ad')
grn.data.write_h5ad(output_file)


# GRN check

In [ ]:
adata_gpu = sc.read_h5ad("Output/GRN_output/test_large_function_fast/split_infer_region_spagrn.h5ad")
adata_gpu

In [ ]:
adata = sc.read_h5ad("Output/GRN_output/test_large_function/split_infer_region_spagrn.h5ad")
adata

In [ ]:
print(f'the number of gpu acceleration is {len(adata_gpu.uns["regulon_dict"].keys())}')
print(f'the number of cpu is {len(adata.uns["regulon_dict"].keys())}')

In [ ]:
len(set(adata_gpu.uns['regulon_dict'].keys())&set(adata.uns['regulon_dict'].keys()))

In [ ]:
from __future__ import annotations
from typing import Dict, Iterable, Optional, Set, Tuple, Union
import numpy as np
import pandas as pd

try:
    import scanpy as sc
    from anndata import AnnData
except Exception:
    sc = None
    AnnData = object  # fallback for typing only

RegulonDict = Dict[str, Set[str]]

def _as_str_set(x) -> Set[str]:
    """Convert an iterable of values to a set of stripped strings, ignoring NaNs."""
    if x is None:
        return set()
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        return {str(v).strip() for v in x if pd.notna(v)}
    # Other iterables
    try:
        return {str(v).strip() for v in x}
    except Exception:
        return set()

def extract_regulon_dict(adata: "AnnData", key: str = "regulon_dict") -> RegulonDict:
    """Extract a regulon dictionary from adata.uns[key], ensuring it's a dict of TF -> set(targets)."""
    if not hasattr(adata, "uns") or key not in adata.uns:
        raise KeyError(f"Key '{key}' not found in AnnData.uns")
    d = adata.uns[key]
    if not isinstance(d, dict):
        raise TypeError(f"uns['{key}'] is not a dict, but {type(d)}")
    out: RegulonDict = {}
    for tf, targets in d.items():
        tf_str = str(tf).strip()
        out[tf_str] = _as_str_set(targets)
    return out

def make_pair_set(regulons: RegulonDict, tfs_subset: Optional[Iterable[str]] = None) -> Set[Tuple[str, str]]:
    """Build a set of (TF, target) pairs, optionally restricting to a subset of TFs."""
    if tfs_subset is None:
        keys = regulons.keys()
    else:
        s = {str(t).strip() for t in tfs_subset}
        keys = [k for k in regulons.keys() if k in s]
    pairs: Set[Tuple[str, str]] = set()
    for tf in keys:
        for tg in regulons.get(tf, set()):
            pairs.add((tf, tg))
    return pairs

def overlap_metrics(pairs_a: Set[Tuple[str, str]], pairs_b: Set[Tuple[str, str]]) -> dict:
    """Compute overlap counts and percentages between two sets of edges."""
    inter = pairs_a & pairs_b
    union = pairs_a | pairs_b
    nA, nB, nI, nU = len(pairs_a), len(pairs_b), len(inter), len(union)

    def pct(x, y):
        return float("nan") if y == 0 else 100.0 * x / y

    return {
        "n_A": nA,
        "n_B": nB,
        "n_intersect": nI,
        "n_union": nU,
        "percent_of_A": pct(nI, nA),               # |A∩B| / |A|
        "percent_of_B": pct(nI, nB),               # |A∩B| / |B|
        "percent_of_union(Jaccard)": pct(nI, nU),  # |A∩B| / |A∪B|
        "percent_of_min": pct(nI, min(nA, nB)) if min(nA, nB) > 0 else float("nan"),
    }

def per_tf_overlap(regA: RegulonDict, regB: RegulonDict, overlapping_tfs: Iterable[str]) -> pd.DataFrame:
    """Compute per-TF target overlap metrics across the overlapping TF set."""
    rows = []
    for tf in overlapping_tfs:
        tA = regA.get(tf, set())
        tB = regB.get(tf, set())
        inter = tA & tB
        union = tA | tB
        nA, nB, nI, nU = len(tA), len(tB), len(inter), len(union)
        jacc = (nI / nU * 100.0) if nU > 0 else np.nan
        pctA = (nI / nA * 100.0) if nA > 0 else np.nan
        pctB = (nI / nB * 100.0) if nB > 0 else np.nan
        rows.append({
            "TF": tf, "targets_A": nA, "targets_B": nB, "overlap": nI,
            "pct_of_A(%)": pctA, "pct_of_B(%)": pctB, "Jaccard(%)": jacc
        })
    return pd.DataFrame(rows).sort_values(["Jaccard(%)", "overlap"], ascending=[False, False])

def ensure_anndata(x: Union[str, "AnnData"]) -> "AnnData":
    """Return an AnnData object; if given a path string, read it from .h5ad."""
    if isinstance(x, str):
        if sc is None:
            raise ImportError("scanpy is required to read .h5ad files; please install scanpy")
        return sc.read_h5ad(x)
    return x

def report_overlap(
    adata_a: Union[str, "AnnData"],
    adata_b: Union[str, "AnnData"],
    key: str = "regulon_dict",
    overlapping_tfs: Optional[Iterable[str]] = None,  # If provided (e.g., the 273 TFs), use it; otherwise auto-intersection
    example_tf: Optional[str] = "ALX3(+)",
    topk_show: int = 10,
) -> dict:
    """Print and return overlap statistics between two regulon dictionaries in AnnData objects."""
    A = ensure_anndata(adata_a)
    B = ensure_anndata(adata_b)

    regA = extract_regulon_dict(A, key=key)
    regB = extract_regulon_dict(B, key=key)

    tfsA, tfsB = set(regA.keys()), set(regB.keys())
    tf_inter = set(overlapping_tfs) if overlapping_tfs is not None else (tfsA & tfsB)

    # All edges (no TF restriction)
    pairsA_all = make_pair_set(regA, None)
    pairsB_all = make_pair_set(regB, None)
    overall_all = overlap_metrics(pairsA_all, pairsB_all)

    # Edges restricted to overlapping TFs (typical use for "273 TFs")
    pairsA_sub = make_pair_set(regA, tf_inter)
    pairsB_sub = make_pair_set(regB, tf_inter)
    overall_sub = overlap_metrics(pairsA_sub, pairsB_sub)

    print("=== Summary ===")
    print(f"A: #TFs={len(tfsA)}, #edges={len(pairsA_all)}")
    print(f"B: #TFs={len(tfsB)}, #edges={len(pairsB_all)}")
    print(f"#Overlapping TFs={len(tf_inter)}")

    print("\n=== Global TF-target overlap (no TF restriction) ===")
    print(f"#Intersect edges={overall_all['n_intersect']}, #Union edges={overall_all['n_union']}")
    print(f"|A∩B|/|A| = {overall_all['percent_of_A']:.2f}%")
    print(f"|A∩B|/|B| = {overall_all['percent_of_B']:.2f}%")
    print(f"Jaccard = {overall_all['percent_of_union(Jaccard)']:.2f}%")

    print("\n=== Overlap restricted to overlapping TFs (recommended; matches the 273 TFs scenario) ===")
    print(f"#A_subset edges={overall_sub['n_A']}, #B_subset edges={overall_sub['n_B']}")
    print(f"#Intersect edges={overall_sub['n_intersect']}, #Union edges={overall_sub['n_union']}")
    print(f"|A∩B|/|A_subset| = {overall_sub['percent_of_A']:.2f}%")
    print(f"|A∩B|/|B_subset| = {overall_sub['percent_of_B']:.2f}%")
    print(f"Jaccard_subset = {overall_sub['percent_of_union(Jaccard)']:.2f}%")

    # Per-TF overlap table
    df_tf = per_tf_overlap(regA, regB, sorted(tf_inter))
    print(f"\n=== Per-TF overlap (Top {topk_show}, sorted by Jaccard desc) ===")
    if not df_tf.empty:
        print(df_tf.head(topk_show).to_string(index=False))
    else:
        print("No overlapping TFs; cannot compute.")

    # Example TF details (e.g., 'ALX3(+)')
    if example_tf is not None and example_tf in tf_inter:
        tA = regA.get(example_tf, set())
        tB = regB.get(example_tf, set())
        inter = tA & tB
        print(f"\n=== Example TF: {example_tf} ===")
        print(f"#targets in A={len(tA)}, #targets in B={len(tB)}, #overlap={len(inter)}")
        preview = sorted(list(inter))[:20]
        print(f"Overlapping targets (first 20): {preview}")
    elif example_tf is not None:
        print(f"\nNote: example TF {example_tf} is not in the overlapping TF set.")

    return {
        "overall_all": overall_all,
        "overall_overlap_TFs": overall_sub,
        "per_tf_df": df_tf,
        "overlapping_tfs": sorted(tf_inter),
    }

if __name__ == "__main__":
    # Usage 1: if you already have two variables `adata_gpu` and `adata`
    # (both containing uns['regulon_dict']) in memory:
    # res = report_overlap(adata_gpu, adata, key="regulon_dict", overlapping_tfs=None, example_tf="ALX3(+)")
    #
    # Usage 2: if they are two .h5ad file paths (requires scanpy installed):
    # res = report_overlap("adata_gpu.h5ad", "adata.h5ad", key="regulon_dict", overlapping_tfs=None, example_tf="ALX3(+)")
    #
    # If you already have the full list of the 273 overlapping TFs, pass it via `overlapping_tfs`:
    # overlap_tfs = [...]  # a list of TF names
    # res = report_overlap(adata_gpu, adata, overlapping_tfs=overlap_tfs, example_tf="ALX3(+)")
    pass

In [ ]:
res = report_overlap(adata_gpu, adata, key="regulon_dict")